In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Settings
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

In [ ]:
BASE_PATH = "/content/drive/My Drive/Datasets/Customer_Seg/outputs"

In [ ]:
rfm_df = pd.read_csv(f"{BASE_PATH}/all_customer_rfm_segmentation.csv")

In [ ]:
PROFIT_MARGIN = 0.10

In [ ]:
rfm_df['Historical_CLV'] = rfm_df['Monetary'] * PROFIT_MARGIN

In [ ]:
historical_clv_results = rfm_df[['customer_unique_id', 'Cluster', 'Monetary', 'Historical_CLV']]
print("--- Historical CLV (Top 5 Customers) ---")
print(historical_clv_results.sort_values(by='Historical_CLV', ascending=False).head())

--- Historical CLV (Top 5 Customers) ---
                     customer_unique_id  Cluster  Monetary  Historical_CLV
3724   0a0a92112bd4c708ca5fde585afaa872        0  13664.08        1366.408
79636  da122df9eeddfedc1dc1f5349a1a690c        0   7571.63         757.163
43168  763c8b1c9c68a0229c42c9fc6f662b93        0   7274.88         727.488
80463  dc4802a71eae9be1dd28f5d788ceb526        0   6929.31         692.931
25436  459bef486812aa25204be022145caa62        0   6922.21         692.221


In [ ]:
OUTPUT_PATH = f"{BASE_PATH}"

import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
rfm_df.to_csv(f"{OUTPUT_PATH}/historical_clv_results.csv", index=False)

#Predictive CLV Implementation

In [ ]:
!pip install lifetimes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 8.1 MB/s eta 0:00:00


In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

In [ ]:
BASE_PATH1 = "/content/drive/My Drive/Datasets/Customer_Seg"

In [ ]:
customers = pd.read_csv(f"{BASE_PATH1}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{BASE_PATH1}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{BASE_PATH1}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{BASE_PATH1}/olist_order_payments_dataset.csv")

In [ ]:
orders = orders[orders["order_status"] == "delivered"]

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)


In [ ]:
data = orders.merge(customers, on="customer_id", how="left")

In [ ]:
data = data.merge(order_items, on="order_id", how="left")

In [ ]:
data = pd.merge(orders, customers, on='customer_id').merge(order_items, on='order_id')
data['revenue'] = data['price'] + data['freight_value']

In [ ]:
# 2. Create the CLV Summary Table (Frequency, Recency, T, Monetary)
# Note: In lifetimes, Recency = time between first and last purchase
# T = age of customer (time since first purchase)
clv_summary = summary_data_from_transaction_data(
    data,
    customer_id_col='customer_unique_id',
    datetime_col='order_purchase_timestamp',
    monetary_value_col='revenue',
    observation_period_end=data['order_purchase_timestamp'].max()
)

In [ ]:
# Filter for repeat customers (Frequency > 0) for the Gamma-Gamma model
clv_summary = clv_summary[clv_summary['frequency'] > 0]

In [ ]:
# 3. Fit the BG/NBD Model (Predicts expected future transactions)
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(clv_summary['frequency'], clv_summary['recency'], clv_summary['T'])

<lifetimes.BetaGeoFitter: fitted with 2015 subjects, a: 0.79, alpha: 148.40, b: 0.09, r: 1.86>

In [ ]:
# 4. Fit the Gamma-Gamma Model (Predicts expected average transaction value)
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(clv_summary['frequency'], clv_summary['monetary_value'])

<lifetimes.GammaGammaFitter: fitted with 2015 subjects, p: 4.12, q: 0.50, v: 3.85>

In [ ]:
# 5. Predict CLV for the next 3 months (Time is in months)
clv_summary['Predictive_CLV_3M'] = ggf.customer_lifetime_value(
    bgf,
    clv_summary['frequency'],
    clv_summary['recency'],
    clv_summary['T'],
    clv_summary['monetary_value'],
    time=3,           # 12 months projection
    discount_rate=0.01 # Monthly discount rate
)

In [ ]:
# 6. Merge with your Cluster info from the RFM CSV
final_clv = clv_summary.reset_index().merge(
    rfm_df[['customer_unique_id', 'Cluster']],
    on='customer_unique_id',
    how='left'
)

In [ ]:
final_clv.to_csv(f"{OUTPUT_PATH}/predictive_clv_output.csv", index=False)